# Churn — Telco

Run All. Não precisa configurar nada.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv(r"C:\Users\gugaa\Downloads\-Telco-Customer-Churn.csv")

# TotalCharges vem como texto, com espaços em branco onde tenure=0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

y = (df["Churn"] == "Yes").astype(int)
X = pd.get_dummies(df.drop(columns=["customerID", "Churn"]), drop_first=True)

print(f"{len(df):,} clientes | {X.shape[1]} features | churn {y.mean():.1%}")
df.head(3)

## Treinar

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

base_model = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                           scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(),
                           eval_metric="logloss", random_state=42)

# calibra as probabilidades (o scale_pos_weight deforma o score)
modelo = CalibratedClassifierCV(base_model, method="isotonic", cv=5)
modelo.fit(Xtr, ytr)
p = modelo.predict_proba(Xte)[:, 1]

# estabilidade: AUC em 5 folds, para saber a incerteza do número
cv = cross_val_score(base_model, Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                     scoring="roc_auc")
print(f"AUC em 5 folds: {cv.mean():.3f} +/- {cv.std():.3f}")

## Métricas gerais

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix,
                             classification_report)

base = yte.mean()
fpr, tpr, _ = roc_curve(yte, p)
ks = np.max(tpr - fpr)

print(f"taxa de churn no teste .. {base:.1%}\n")
print(f"ROC-AUC ................. {roc_auc_score(yte, p):.3f}")
print(f"PR-AUC .................. {average_precision_score(yte, p):.3f}"
      f"   (aleatório = {base:.3f})")
print(f"KS ...................... {ks:.3f}")
print(f"Brier ................... {brier_score_loss(yte, p):.4f}")

**ROC-AUC** — capacidade de ordenar. 0.5 é moeda, 0.85 é bom para churn.
**PR-AUC** — o que importa quando o alvo é raro; compare sempre com a taxa base.
**KS** — separação entre quem fica e quem sai. Acima de 0.30 é usável.
**Brier** — erro da probabilidade. Quanto menor melhor.

## Onde cortar

In [ ]:
# opção 1: threshold que maximiza F1
prec, rec, thr = precision_recall_curve(yte, p)
f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
k = int(np.nanargmax(f1[:-1]))
thr_f1 = thr[k]

print(f"THRESHOLD POR F1: {thr_f1:.3f}")
print(f"  precision={prec[k]:.1%}  recall={rec[k]:.1%}  F1={f1[k]:.3f}\n")

# opção 2: threshold pela capacidade do time (aqui, 20% da base)
CAPACIDADE = 0.20
n = int(np.ceil(CAPACIDADE * len(yte)))
thr_cap = np.sort(p)[::-1][n - 1]
sel = p >= thr_cap
prec_cap = yte[sel].mean()

print(f"THRESHOLD POR CAPACIDADE ({CAPACIDADE:.0%} da base): {thr_cap:.3f}")
print(f"  aborda {sel.sum():,} clientes")
print(f"  {prec_cap:.1%} deles realmente cancelam  (lift {prec_cap/base:.2f}x)")
print(f"  cobre {yte[sel].sum()/yte.sum():.1%} de todos os cancelamentos\n")

tn, fp, fn, tp = confusion_matrix(yte, sel.astype(int)).ravel()
print("matriz de confusão no threshold de capacidade")
print(f"                 previsto: fica    previsto: churn")
print(f"  real: fica  {tn:>14,} {fp:>18,}")
print(f"  real: churn {fn:>14,} {tp:>18,}\n")
print(classification_report(yte, sel.astype(int), target_names=["fica", "churn"], digits=3))

Os dois cortes respondem perguntas diferentes. O de F1 é o equilíbrio matemático.
O de capacidade é o que a operação consegue executar: se o time liga para 20% da base
por mês, o threshold é esse, e o número que importa é o lift.

## Decis

In [ ]:
d = pd.DataFrame({"y": yte.values, "p": p}).sort_values("p", ascending=False).reset_index(drop=True)
d["decil"] = pd.qcut(d.index, 10, labels=range(1, 11)).astype(int)

tab = d.groupby("decil").agg(clientes=("y", "size"), churns=("y", "sum"),
                             taxa=("y", "mean"), score_medio=("p", "mean"))
tab["lift"] = (tab["taxa"] / base).round(2)
tab["% churn capturado"] = (100 * tab["churns"].cumsum() / d["y"].sum()).round(1)
tab["taxa"] = (100 * tab["taxa"]).round(1)
tab["score_medio"] = tab["score_medio"].round(3)

tab.style.background_gradient(subset=["lift"], cmap="Blues")

Decil 1 = os 10% de maior score. **Se o lift do decil 1 não passar de 2x, o modelo
está pouco acima de sortear clientes.** A coluna `% churn capturado` é a que vai
para a apresentação: abordando até o decil N, você pega X% dos cancelamentos.

## Calibração

In [ ]:
cal = pd.DataFrame({"y": yte.values, "p": p})
cal["faixa"] = pd.qcut(cal["p"].rank(method="first"), 10, labels=False)
g = cal.groupby("faixa").agg(clientes=("y", "size"), previsto=("p", "mean"),
                             observado=("y", "mean")).round(3)
g["erro"] = (g["observado"] - g["previsto"]).round(3)

pior = g["erro"].abs().max()
print(f"maior desvio: {pior:.3f}", "(ok)" if pior < 0.10 else "(mal calibrado)")
print("Se estiver ok, o score pode ser lido como probabilidade de verdade.")
print("Se não, use só o ranking, nunca o número em conta financeira.\n")
g

## Gráficos

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

ax[0,0].plot(fpr, tpr, color="#4C6EF5", lw=2, label=f"AUC = {roc_auc_score(yte, p):.3f}")
ax[0,0].plot([0,1],[0,1],"--",color="#adb5bd")
ax[0,0].set_title("Curva ROC"); ax[0,0].set_xlabel("falso positivo")
ax[0,0].set_ylabel("verdadeiro positivo"); ax[0,0].legend()

ax[0,1].bar(tab.index, tab["lift"], color="#4C6EF5")
ax[0,1].axhline(1, ls="--", color="#adb5bd")
ax[0,1].set_title("Lift por decil"); ax[0,1].set_xlabel("decil (1 = maior score)")

ax[1,0].plot([0]+list(np.arange(1,11)*10), [0]+list(tab["% churn capturado"]),
             "o-", color="#4C6EF5", label="modelo")
ax[1,0].plot([0,100],[0,100],"--",color="#adb5bd", label="aleatório")
ax[1,0].set_title("Ganho acumulado"); ax[1,0].set_xlabel("% da base abordada")
ax[1,0].set_ylabel("% do churn capturado"); ax[1,0].legend()

ax[1,1].plot(g["previsto"], g["observado"], "o-", color="#4C6EF5")
lim = max(g["previsto"].max(), g["observado"].max())*1.1
ax[1,1].plot([0,lim],[0,lim],"--",color="#adb5bd")
ax[1,1].set_title("Calibração"); ax[1,1].set_xlabel("previsto"); ax[1,1].set_ylabel("observado")

plt.tight_layout(); plt.show()

## O que puxa o churn

In [ ]:
base_model.fit(Xtr, ytr)
imp = pd.Series(base_model.feature_importances_, index=X.columns).sort_values(ascending=False)

imp.head(15).iloc[::-1].plot.barh(figsize=(8, 5), color="#4C6EF5",
                                  title="Features mais importantes")
plt.tight_layout(); plt.show()

print(imp.head(10).round(4).to_string())

## Salvar os scores

In [ ]:
saida = pd.DataFrame({
    "customerID": df.loc[Xte.index, "customerID"].values,
    "churn_real": yte.values,
    "score": p.round(4),
}).sort_values("score", ascending=False)

saida["decil"] = pd.qcut(saida["score"].rank(method="first", ascending=False),
                         10, labels=range(1, 11)).astype(int)
saida.to_csv("scores_churn.csv", index=False)

print(f"{len(saida):,} clientes salvos em scores_churn.csv")
saida.head(10)